# Bounding Box Tagging

## Tracking

First of all, we have to retrive from a video the tracking of the objects. In order to do that we used the following project: [Yolov5_StrongSORT_OSNet](https://github.com/mikel-brostrom/Yolov5_StrongSORT_OSNet.git)

You can find the codes inside **./src/pcgvs/extraction**.

Now we use `extract-tubes`, this function uses *Yolov5_StrongSORT_OSNet* to extract the tubes. A tube is relative to a single object in the scene and refers to its position in time, so it's a triplet: $x$, $y$ and $t$.

In [ ]:
from pcgvs.extraction import extract_tubes
extract_tubes(source="../Users/nguyenduy/Desktop/pcgvs-main/notebooks/Metadata/Video_input/*.mp4", outputdir='/Users/nguyenduy/Desktop/pcgvs-main/notebooks/Metadata/synopsis')

After, we get the following file **./synopsis/tubes/exp/tracks/video-1-raw.txt**. Let's create a dictionary whose index is the frame number. A record contains a list of objects that are within that frame and the following information is saved for each object:

- **tag**: identifier of the object;
- **x**: coordinate along the x axis;
- **y**: coordinate along the y axis;
- **w**: width of the bounding box;
- **h**: height of the bounding box.

In [1]:
frames = {}
with open("/Users/nguyenduy/Desktop/pcgvs-main/notebooks/Metadata/data_input_txt_object/trungquoc.txt", 'r') as f:
    for line in f:
        f, id, x, y, w, h = line.split()[0:6]
        if int(f) not in frames.keys():
            frames[int(f)] = []
        frames[int(f)].append([id, int(x), int(y), int(w), int(h)])
        
frames

{169: [['1', 40, 242, 25, 21]],
 170: [['1', 39, 240, 27, 24]],
 171: [['1', 39, 239, 27, 25]],
 174: [['1', 37, 238, 30, 26]],
 175: [['1', 38, 236, 25, 27]],
 176: [['1', 36, 234, 29, 30], ['4', 108, 244, 23, 20]],
 177: [['1', 35, 232, 33, 31], ['4', 109, 244, 24, 20]],
 178: [['1', 34, 231, 34, 33], ['4', 112, 241, 23, 22]],
 179: [['1', 35, 231, 36, 33], ['4', 112, 240, 25, 23]],
 180: [['1', 39, 232, 32, 32], ['4', 113, 238, 25, 25]],
 181: [['1', 42, 231, 30, 33], ['4', 113, 236, 25, 27]],
 182: [['1', 43, 230, 32, 33], ['4', 112, 235, 27, 29]],
 183: [['1', 44, 228, 33, 35], ['4', 111, 235, 26, 29]],
 184: [['1', 37, 225, 44, 38], ['4', 109, 235, 28, 28]],
 185: [['1', 38, 223, 43, 40], ['4', 104, 235, 33, 29]],
 186: [['1', 46, 222, 37, 42], ['4', 101, 232, 39, 32]],
 187: [['1', 47, 222, 38, 42], ['4', 100, 229, 42, 35]],
 188: [['1', 47, 222, 38, 42], ['4', 99, 228, 44, 36]],
 189: [['1', 46, 222, 37, 42], ['4', 99, 227, 44, 37]],
 190: [['1', 40, 220, 44, 43], ['4', 100, 22

This code takes as input the video, to which we want to add the bounding boxes, and the output folder. The dictionary created previously is used to locate the bounding boxes.

In [2]:
import cv2
import os

def start_detection(source, outputdir):
    
    cap = cv2.VideoCapture(source)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    
    out = cv2.VideoWriter(os.path.join(outputdir, '/Users/nguyenduy/Desktop/pcgvs-main/notebooks/Metadata/Video_input/video4.mp4'),fourcc, 30, (1280,1080))
    
    ret = True
    emergencybg = None
    background = True
    count_bg = 0
    num_frame = 1
    while ret:
        ret, frame = cap.read()
        
        if emergencybg is None and frame is not None:
            emergencybg = frame
        
        if num_frame in frames.keys():
            count_bg = 0
            for id, x, y, w, h in frames[num_frame]:
                cv2.rectangle(frame,(x,y),(x+w,y+h),thickness=2,color=(255,0,0))
                cv2.putText(frame,str(id),(x,y-20),cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
        elif background and count_bg > 8:
            background = False
            cv2.imwrite(os.path.join(outputdir, 'background.jpg'), frame)
        else:
            count_bg += 1
        
        num_frame += 1

        try:
            out.write(frame[:-200])
        except:
            continue
    
    if count_bg <= 10 and background:
        cv2.imwrite(os.path.join(outputdir, 'background.jpg'), emergencybg)
            
    out.release()

In [ ]:
# start_detection(source="/Users/nguyenduy/Desktop/pcgvs-main/notebooks/Metadata/Video_input/test2.mp4", outputdir="./synopsis")